# CSCE 636 Project 1 — Grace HPC (Scaled-Up)

**Scaled-up MoE training pipeline for TAMU Grace A100 GPU.**

| Component | Local (notebook) | Grace (this notebook) |
|-----------|-----------------|----------------------|
| Easy group arch | 256h / 3 blocks | **512h / 5 blocks** |
| Medium group arch | 384 / 4 blocks | **768 / 6 blocks** |
| Hard group arch | 512 / 5 blocks | **1024 / 8 blocks** |
| Base runs (easy) | 2 | **5** |
| Base runs (hard) | 3 | **10** |
| Retrain runs | 5 | **10** |
| Ensemble size | 8 models | **20 models** |
| Ensemble archs | 512–768h | **768–2048h, 4–10 blocks** |
| Base epochs | 300–500 | **800** |
| Weak/ensemble epochs | 600 | **1000** |
| LP weak | 8K/group | **20K/group** |
| Batch size | 256 | **512** |

In [3]:
# ============================================================
# Section 0: Configuration
# ============================================================
# Edit these before running on Grace.

DATA_DIR = '.'          # directory with CSCE-636-Project-1-Train-* files
OUTPUT_DIR = '.'        # directory for moe_model.pth + predicted_mHeights
SEED = 42

# --- Scaled-up hyperparams ---
BASE_EPOCHS = 800       # max epochs for base experts
WEAK_EPOCHS = 1000      # max epochs for retrained weak-group experts
ENSEMBLE_EPOCHS = 1000  # max epochs per ensemble member
BASE_RUNS = 5           # best-of-N for easy base experts
HARD_RUNS = 10          # best-of-N for hard base experts
RETRAIN_RUNS = 10       # best-of-N for retrained experts
ENSEMBLE_SIZE = 20      # ensemble members per weak group
LP_PER_GROUP = 5000     # initial LP augmentation per group
LP_WEAK = 20000         # extra LP samples per weak group
BATCH_SIZE = 512        # larger for GPU
COST_THRESHOLD = 1.0    # groups above this are "weak"
NUM_WORKERS = 4         # DataLoader workers

In [4]:
# ============================================================
# Section 1: Imports & Device Setup
# ============================================================
import os, sys, time, pickle, math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.optimize import linprog
from itertools import combinations
from collections import Counter, defaultdict
from torch.optim.swa_utils import AveragedModel, SWALR
import warnings
warnings.filterwarnings('ignore')

np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
    print(f'[GPU] {torch.cuda.get_device_name(0)}')
    print(f'[GPU] Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

Device: mps


In [5]:
# ============================================================
# Section 2: Architecture + Helpers
# ============================================================

class MHeightDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.block(x) + x)

class ExpertDNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=3, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
        )
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)]
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1),
        )
    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_blocks(x)
        return self.output_head(x)

# --- Loss helpers ---
def log_cosh_loss(pred, target):
    diff = pred - target
    return torch.mean(torch.log(torch.cosh(diff + 1e-12)))

def feature_noise(x, std=0.05):
    return x + torch.randn_like(x) * std

def combined_loss(pred, target, beta=0.5):
    return (0.5 * nn.functional.smooth_l1_loss(pred, target, beta=beta)
            + 0.3 * nn.functional.mse_loss(pred, target)
            + 0.2 * log_cosh_loss(pred, target))

# --- Training helper ---
def train_single_run(model, train_loader, val_loader, train_size, val_size,
                     device, lr, weight_decay, max_epochs, patience,
                     noise_std=0.0, use_combined_loss=False, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if use_combined_loss:
        criterion_fn = lambda p, t: combined_loss(p, t, beta=0.5)
    else:
        criterion = nn.SmoothL1Loss(beta=0.5)
        criterion_fn = lambda p, t: criterion(p, t)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=60, T_mult=2, eta_min=1e-7)
    best_val = float('inf')
    pat_cnt = 0
    best_state = None
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            if noise_std > 0:
                bx = feature_noise(bx, std=noise_std)
            pred = model(bx)
            loss = criterion_fn(pred, by)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * bx.size(0)
        train_loss /= train_size
        scheduler.step()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                val_loss += nn.functional.mse_loss(model(bx), by).item() * bx.size(0)
        val_loss /= val_size
        if val_loss < best_val:
            best_val = val_loss
            pat_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            pat_cnt += 1
        if (epoch + 1) % 200 == 0 or epoch == 0:
            print(f'      Ep [{epoch+1:4d}/{max_epochs}] Train: {train_loss:.6f} '
                  f'Val: {val_loss:.6f} Best: {best_val:.6f}')
        if pat_cnt >= patience:
            print(f'      Early stop ep {epoch+1}, best: {best_val:.6f}')
            break
    return best_val, best_state

def batch_predict(model, tensor, batch_size=2048):
    preds = []
    model.eval()
    with torch.no_grad():
        for bi in range(0, len(tensor), batch_size):
            preds.append(model(tensor[bi:bi+batch_size]).cpu().numpy().flatten())
    return np.concatenate(preds)

print('Architecture + helpers defined ✓')

Architecture + helpers defined ✓


In [6]:
# ============================================================
# Section 3: LP m-height + Feature Engineering (77-dim)
# ============================================================

def compute_m_height_lp(n, k, m, P):
    G = np.hstack([np.eye(k), P])
    max_z = 1.0
    for S in combinations(range(n), m):
        S_set = set(S)
        S_bar = [t for t in range(n) if t not in S_set]
        for j in S:
            c = -G[:, j]
            A_ub, b_ub = [], []
            for t in S_bar:
                A_ub.append(G[:, t]);  b_ub.append(1.0)
                A_ub.append(-G[:, t]); b_ub.append(1.0)
            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), method='highs')
            if result.success:
                z = -result.fun
                if z > max_z:
                    max_z = z
    return max_z

def generate_lp_samples(n, k, m, num_samples, p_ranges=None):
    if p_ranges is None:
        p_ranges = [1, 2, 5, 10, 20, 50, 100, 200]
    new_X, new_y = [], []
    per_range = max(1, num_samples // len(p_ranges))
    for p_range in p_ranges:
        for _ in range(per_range):
            P = np.random.uniform(-p_range, p_range, size=(k, n - k))
            mh = compute_m_height_lp(n, k, m, P)
            if np.isfinite(mh) and mh >= 1.0:
                new_X.append([n, k, m, P])
                new_y.append(mh)
    return new_X, new_y

FEATURE_DIM = 77

def extract_features(sample):
    n, k, m, P = sample
    max_k, max_nk, max_sv = 6, 5, 5
    P_padded = np.zeros((max_k, max_nk))
    P_padded[:k, :n-k] = P
    P_flat = P_padded.flatten()
    row_norms = np.zeros(max_k)
    for i in range(k): row_norms[i] = np.linalg.norm(P[i, :])
    col_norms = np.zeros(max_nk)
    for j in range(n - k): col_norms[j] = np.linalg.norm(P[:, j])
    sv = np.zeros(max_sv)
    try:
        svs = np.linalg.svd(P, compute_uv=False)
        sv[:len(svs)] = np.sort(svs)[::-1]
    except: pass
    frob_norm = np.linalg.norm(P, 'fro')
    max_abs = np.max(np.abs(P))
    mean_abs = np.mean(np.abs(P))
    std_abs = np.std(np.abs(P))
    min_abs = np.min(np.abs(P))
    try: cond = np.log1p(np.linalg.cond(P))
    except: cond = 0.0
    if not np.isfinite(cond): cond = 50.0
    mean_row = np.mean(row_norms[:k]) if k > 0 else 1.0
    mean_col = np.mean(col_norms[:n-k]) if (n-k) > 0 else 1.0
    ratio = mean_row / (mean_col + 1e-8)
    G = np.hstack([np.eye(k), P])
    g_col_norms = np.zeros(9)
    for j in range(n): g_col_norms[j] = np.linalg.norm(G[:, j])
    g_col_max = np.max(g_col_norms)
    g_col_min = np.min(g_col_norms[:n])
    g_col_mean = np.mean(g_col_norms[:n])
    g_col_std = np.std(g_col_norms[:n])
    try: rank = np.linalg.matrix_rank(P)
    except: rank = min(k, n-k)
    effective_rank = np.sum(sv > 1e-6)
    sv_ratio = sv[0] / (sv[min(k, n-k)-1] + 1e-10) if sv[min(k, n-k)-1] > 1e-10 else 100.0
    sv_sum = np.sum(sv)
    sv_energy_ratio = sv[0] / (sv_sum + 1e-10)
    try:
        PtP = P.T @ P; ptP_trace = np.trace(PtP); ptP_frob = np.linalg.norm(PtP, 'fro')
    except:
        ptP_trace = frob_norm**2; ptP_frob = frob_norm**2
    try:
        sq = min(k, n-k); det_val = np.log1p(abs(np.linalg.det(P[:sq, :sq])))
    except: det_val = 0.0
    if not np.isfinite(det_val): det_val = 0.0
    return np.concatenate([
        [n, k, m], P_flat, row_norms, col_norms, sv,
        [frob_norm, max_abs, mean_abs, std_abs, min_abs], [cond, ratio],
        g_col_norms, [g_col_max, g_col_min, g_col_mean, g_col_std],
        [rank, effective_rank], [sv_ratio, sv_sum, sv_energy_ratio],
        [ptP_trace, ptP_frob], [det_val],
    ])

print(f'Feature dim: {FEATURE_DIM}, LP + feature functions defined ✓')

Feature dim: 77, LP + feature functions defined ✓


In [ ]:
# ============================================================
# Section 4: Load Data + Initial LP Augmentation
# ============================================================
import os

x_path = os.path.join(DATA_DIR, 'CSCE-636-Project-1-Train-n_k_m_P')
y_path = os.path.join(DATA_DIR, 'CSCE-636-Project-1-Train-mHeights')
with open(x_path, 'rb') as f: X_raw = pickle.load(f)
with open(y_path, 'rb') as f: y_raw = pickle.load(f)
print(f'Loaded {len(X_raw)} samples, y range [{min(y_raw):.4f}, {max(y_raw):.4f}]')

param_counts = Counter([(s[0], s[1], s[2]) for s in X_raw])
for params, count in sorted(param_counts.items()):
    print(f'  n={params[0]}, k={params[1]}, m={params[2]}: {count}')

# --- LP augmentation ---
X_combined = list(X_raw)
y_combined = list(y_raw)
target_per_group = 15000

print('\nLP augmentation...')
for (n_val, k_val, m_val), count in sorted(param_counts.items()):
    deficit = max(0, target_per_group - count)
    num_gen = min(deficit, LP_PER_GROUP)
    if num_gen > 0:
        print(f'  (n={n_val},k={k_val},m={m_val}): +{num_gen}...', end=' ', flush=True)
        t0 = time.time()
        new_x, new_y = generate_lp_samples(n_val, k_val, m_val, num_gen,
                                            p_ranges=[1, 2, 5, 10, 20, 50, 100])
        X_combined.extend(new_x); y_combined.extend(new_y)
        print(f'{len(new_y)} valid ({time.time()-t0:.1f}s)')
    else:
        print(f'  (n={n_val},k={k_val},m={m_val}): {count} — skip')

print(f'\nTotal after augmentation: {len(X_combined)}')

Loaded 96524 samples, y range [2.0000, 6450061.6667]
  n=9, k=4, m=2: 11157
  n=9, k=4, m=3: 11003
  n=9, k=4, m=4: 10253
  n=9, k=4, m=5: 6765
  n=9, k=5, m=2: 10803
  n=9, k=5, m=3: 10159
  n=9, k=5, m=4: 6573
  n=9, k=6, m=2: 18898
  n=9, k=6, m=3: 10913

LP augmentation...
  (n=9,k=4,m=2): +3843... 3843 valid (62.9s)
  (n=9,k=4,m=3): +3997... 3997 valid (232.0s)
  (n=9,k=4,m=4): +4747... 

In [ ]:
# ============================================================
# Section 5: Feature Engineering + Per-Group Dataset Prep
# ============================================================
print('Extracting 77-dim features...')
X_array = np.array([extract_features(s) for s in X_combined], dtype=np.float32)
y_log2 = np.log2(np.array(y_combined, dtype=np.float32)).reshape(-1, 1)
X_array = np.nan_to_num(X_array, nan=0.0, posinf=100.0, neginf=-100.0)
print(f'Features: {X_array.shape}, log2(y) range: [{y_log2.min():.4f}, {y_log2.max():.4f}]')

# Per-group splits
group_keys = sorted(set((s[1], s[2]) for s in X_combined))
print(f'Groups: {group_keys}')

group_data = {}
for (k_val, m_val) in group_keys:
    indices = [i for i, s in enumerate(X_combined) if s[1] == k_val and s[2] == m_val]
    X_grp, y_grp = X_array[indices], y_log2[indices]
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_grp)
    np.random.seed(42)
    perm = np.random.permutation(len(X_sc))
    split = int(0.9 * len(perm))
    train_idx, val_idx = perm[:split], perm[split:]
    train_ds = MHeightDataset(X_sc[train_idx], y_grp[train_idx])
    val_ds = MHeightDataset(X_sc[val_idx], y_grp[val_idx])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    group_data[(k_val, m_val)] = {
        'scaler': scaler, 'train_loader': train_loader, 'val_loader': val_loader,
        'train_size': len(train_idx), 'val_size': len(val_idx),
        'feature_dim': X_sc.shape[1],
    }
    print(f'  (k={k_val},m={m_val}): {len(indices)} total, '
          f'{len(train_idx)} train, {len(val_idx)} val')

In [ ]:
# ============================================================
# Section 6: Train 9 Base Experts (SCALED UP)
# ============================================================

GROUP_HPARAMS = {
    # (hidden_dim, num_blocks, dropout, lr, epochs, patience, num_runs)
    (4, 2): (512,  5, 0.15, 8e-4, BASE_EPOCHS, 40, BASE_RUNS),
    (4, 3): (512,  5, 0.15, 8e-4, BASE_EPOCHS, 40, BASE_RUNS),
    (5, 2): (512,  5, 0.15, 8e-4, BASE_EPOCHS, 40, BASE_RUNS),
    (6, 2): (512,  5, 0.15, 8e-4, BASE_EPOCHS, 40, BASE_RUNS),
    (4, 4): (768,  6, 0.12, 5e-4, BASE_EPOCHS, 50, BASE_RUNS),
    (5, 3): (768,  6, 0.12, 5e-4, BASE_EPOCHS, 50, BASE_RUNS),
    (4, 5): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 60, HARD_RUNS),
    (5, 4): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 60, HARD_RUNS),
    (6, 3): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 60, HARD_RUNS),
}

experts = {}
expert_hparams = {}
t_total = time.time()

for gk in group_keys:
    hparams = GROUP_HPARAMS.get(gk, (512, 5, 0.15, 8e-4, BASE_EPOCHS, 40, BASE_RUNS))
    hidden_dim, num_blocks, dropout, lr, max_epochs, patience, num_runs = hparams
    expert_hparams[gk] = (hidden_dim, num_blocks, dropout)
    info = group_data[gk]

    print(f'\n{"="*60}')
    print(f'Expert (k={gk[0]},m={gk[1]}): h={hidden_dim}, b={num_blocks}, '
          f'd={dropout}, runs={num_runs}')
    print(f'  Data: {info["train_size"]} train, {info["val_size"]} val')
    print(f'{"="*60}')

    global_best_val = float('inf')
    global_best_state = None

    for run in range(num_runs):
        print(f'\n    --- Run {run+1}/{num_runs} ---')
        seed = SEED + run * 777 + gk[0] * 100 + gk[1] * 10
        model = ExpertDNN(info['feature_dim'], hidden_dim, num_blocks, dropout).to(device)
        val_loss, state = train_single_run(
            model, info['train_loader'], info['val_loader'],
            info['train_size'], info['val_size'], device,
            lr=lr, weight_decay=1e-4, max_epochs=max_epochs,
            patience=patience, noise_std=0.0,
            use_combined_loss=False, seed=seed,
        )
        print(f'    Run {run+1} best val: {val_loss:.6f}')
        if val_loss < global_best_val:
            global_best_val = val_loss
            global_best_state = state

    model = ExpertDNN(info['feature_dim'], hidden_dim, num_blocks, dropout).to(device)
    model.load_state_dict(global_best_state)
    model.eval()
    experts[gk] = model
    print(f'  ✓ (k={gk[0]},m={gk[1]}): best_val = {global_best_val:.6f}')

print(f'\n[Base experts] Total time: {time.time()-t_total:.1f}s')

In [ ]:
# ============================================================
# Section 7: Evaluate Base → Identify Weak Groups
# ============================================================

group_costs = {}
total_cost_sum, total_count = 0.0, 0
for gk in group_keys:
    model = experts[gk]; model.eval()
    preds_l, trues_l = [], []
    with torch.no_grad():
        for bx, by in group_data[gk]['val_loader']:
            bx = bx.to(device)
            preds_l.append(model(bx).cpu().numpy())
            trues_l.append(by.numpy())
    pa = np.concatenate(preds_l).flatten()
    ta = np.concatenate(trues_l).flatten()
    cost = ((ta - pa) ** 2).mean()
    group_costs[gk] = cost
    total_cost_sum += ((ta - pa) ** 2).sum()
    total_count += len(ta)
    marker = '★ WEAK' if cost > COST_THRESHOLD else '✓ ok'
    print(f'  (k={gk[0]},m={gk[1]}): cost={cost:.4f} {marker}')

overall_base = total_cost_sum / total_count
print(f'\nOverall base cost: {overall_base:.6f}')

weak_groups = [gk for gk in group_keys if group_costs[gk] > COST_THRESHOLD]
print(f'Weak groups: {weak_groups}')

In [ ]:
# ============================================================
# Section 8: Extra LP Augmentation for Weak Groups
# ============================================================

weak_extra_X, weak_extra_y = {}, {}
if weak_groups:
    for gk in weak_groups:
        k_val, m_val = gk
        print(f'Generating {LP_WEAK} LP samples for (k={k_val},m={m_val})...', flush=True)
        t0 = time.time()
        new_x, new_y = generate_lp_samples(
            9, k_val, m_val, LP_WEAK,
            p_ranges=[1, 2, 5, 10, 20, 50, 100, 200]
        )
        weak_extra_X[gk] = new_x
        weak_extra_y[gk] = new_y
        print(f'  Generated {len(new_y)} samples in {time.time()-t0:.1f}s')
else:
    print('No weak groups — skipping extra LP augmentation.')

In [ ]:
# ============================================================
# Section 9: Augmented Datasets + Cross-Group Transfer
# ============================================================

cross_group_map = {
    (4, 5): [(4, 4), (4, 3)],
    (5, 4): [(5, 3), (5, 2)],
    (6, 3): [(6, 2)],
}

weak_group_data = {}
if weak_groups:
    for gk in weak_groups:
        k_val, m_val = gk
        orig_indices = [i for i, s in enumerate(X_combined) if s[1] == k_val and s[2] == m_val]
        X_orig, y_orig = X_array[orig_indices], y_log2[orig_indices]
        extra_feats = np.array([extract_features(s) for s in weak_extra_X[gk]], dtype=np.float32)
        extra_y = np.log2(np.array(weak_extra_y[gk], dtype=np.float32)).reshape(-1, 1)

        cross_X_list, cross_y_list = [], []
        for neighbor_gk in cross_group_map.get(gk, []):
            nk, nm = neighbor_gk
            nidx = [i for i, s in enumerate(X_combined) if s[1] == nk and s[2] == nm]
            if nidx:
                np.random.seed(42)
                n_borrow = max(500, len(nidx) // 10)
                chosen = np.random.choice(nidx, size=min(n_borrow, len(nidx)), replace=False)
                cross_X_list.append(X_array[chosen])
                cross_y_list.append(y_log2[chosen])
                print(f'  Borrowed {len(chosen)} from (k={nk},m={nm})')

        parts_X = [X_orig, extra_feats] + cross_X_list
        parts_y = [y_orig, extra_y] + cross_y_list
        X_all = np.nan_to_num(np.vstack(parts_X), nan=0.0, posinf=100.0, neginf=-100.0)
        y_all = np.vstack(parts_y)

        y_p5, y_p95 = np.percentile(y_all, 2), np.percentile(y_all, 98)
        y_clipped = np.clip(y_all, y_p5, y_p95)
        n_clipped = np.sum(y_all != y_clipped)
        if n_clipped > 0:
            print(f'  Clipped {n_clipped} targets to [{y_p5:.2f}, {y_p95:.2f}]')

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_all)

        n_orig = len(X_orig)
        np.random.seed(42)
        orig_perm = np.random.permutation(n_orig)
        orig_val_size = int(0.1 * n_orig)
        orig_val_idx = orig_perm[:orig_val_size]
        orig_train_idx = orig_perm[orig_val_size:]
        train_indices = list(orig_train_idx) + list(range(n_orig, len(X_scaled)))
        val_indices = list(orig_val_idx)

        train_ds = MHeightDataset(X_scaled[train_indices], y_clipped[train_indices])
        val_ds = MHeightDataset(X_scaled[val_indices], y_all[val_indices])
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  drop_last=True, num_workers=NUM_WORKERS, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)

        weak_group_data[gk] = {
            'scaler': scaler, 'train_loader': train_loader, 'val_loader': val_loader,
            'train_size': len(train_indices), 'val_size': len(val_indices),
            'feature_dim': X_scaled.shape[1],
        }
        cross_cnt = sum(len(cx) for cx in cross_X_list) if cross_X_list else 0
        print(f'  (k={k_val},m={m_val}): {n_orig} orig + {len(extra_feats)} LP + '
              f'{cross_cnt} cross = {len(X_all)} total, '
              f'train={len(train_indices)}, val={len(val_indices)}')
else:
    print('No weak groups — skipping.')

In [ ]:
# ============================================================
# Section 10: Retrain Base Experts for Weak Groups (SCALED UP)
# ============================================================

RETRAIN_CONFIGS = {
    # (hidden, blocks, dropout, wd, lr, epochs, patience, n_runs, noise_std)
    (4, 5): (1024, 8, 0.18, 5e-4, 2e-4, WEAK_EPOCHS, 80, RETRAIN_RUNS, 0.06),
    (5, 4): (1024, 8, 0.18, 5e-4, 2e-4, WEAK_EPOCHS, 80, RETRAIN_RUNS, 0.06),
    (6, 3): (1024, 8, 0.15, 3e-4, 2e-4, WEAK_EPOCHS, 80, RETRAIN_RUNS, 0.05),
}

retrained_experts = {}
retrained_scalers = {}

if weak_groups:
    for gk in weak_groups:
        h_dim, n_blk, drop, wd, lr, max_ep, patience, n_runs, noise_std = RETRAIN_CONFIGS[gk]
        info = weak_group_data[gk]

        print(f'\n{"="*60}')
        print(f'RETRAIN (k={gk[0]},m={gk[1]}): h={h_dim}, b={n_blk}, runs={n_runs}')
        print(f'  Data: {info["train_size"]} train, {info["val_size"]} val')
        print(f'  Old cost: {group_costs[gk]:.4f}')
        print(f'{"="*60}')

        global_best_val = float('inf')
        global_best_state = None

        for run in range(n_runs):
            print(f'\n    --- Run {run+1}/{n_runs} ---')
            seed = 1000 + run * 333 + gk[0] * 100 + gk[1] * 10
            model = ExpertDNN(info['feature_dim'], h_dim, n_blk, drop).to(device)
            val_loss, state = train_single_run(
                model, info['train_loader'], info['val_loader'],
                info['train_size'], info['val_size'], device,
                lr=lr, weight_decay=wd, max_epochs=max_ep,
                patience=patience, noise_std=noise_std,
                use_combined_loss=True, seed=seed,
            )
            print(f'    Run {run+1} best val: {val_loss:.6f}')
            if val_loss < global_best_val:
                global_best_val = val_loss
                global_best_state = state

        model = ExpertDNN(info['feature_dim'], h_dim, n_blk, drop).to(device)
        model.load_state_dict(global_best_state)
        model.eval()
        retrained_experts[gk] = model
        retrained_scalers[gk] = info['scaler']
        print(f'  ✓ Best retrained val: {global_best_val:.6f}')
else:
    print('No weak groups — skipping retrain.')

In [ ]:
# ============================================================
# Section 11: Diverse Ensemble per Weak Group (SCALED UP)
# ============================================================

arch_pool = [
    # (hidden_dim, num_blocks, dropout, lr, noise_std)
    (768,  6,  0.18, 2e-4, 0.06),
    (1024, 5,  0.18, 2e-4, 0.07),
    (768,  8,  0.15, 1e-4, 0.06),
    (1024, 6,  0.15, 2e-4, 0.06),
    (1280, 5,  0.18, 1e-4, 0.05),
    (768,  10, 0.12, 1e-4, 0.06),
    (1024, 8,  0.15, 1e-4, 0.05),
    (1536, 5,  0.18, 1e-4, 0.05),
    (1024, 10, 0.12, 1e-4, 0.05),
    (2048, 4,  0.20, 1e-4, 0.06),
    (768,  7,  0.18, 2e-4, 0.07),
    (1280, 6,  0.15, 1e-4, 0.05),
    (1024, 7,  0.15, 2e-4, 0.06),
    (1536, 6,  0.15, 1e-4, 0.05),
    (2048, 5,  0.18, 1e-4, 0.05),
    (768,  9,  0.12, 1e-4, 0.06),
    (1280, 8,  0.12, 1e-4, 0.05),
    (1536, 4,  0.20, 2e-4, 0.06),
    (2048, 6,  0.15, 1e-4, 0.04),
    (1024, 9,  0.12, 1e-4, 0.05),
]
ENSEMBLE_CONFIGS = []
for i in range(ENSEMBLE_SIZE):
    cfg = arch_pool[i % len(arch_pool)]
    ENSEMBLE_CONFIGS.append((*cfg, (i + 1) * 100))

weak_ensembles = {}
weak_scalers = {}
ensemble_val_costs = {}

if weak_groups:
    for gk in weak_groups:
        info = weak_group_data[gk]
        weak_scalers[gk] = info['scaler']
        ens_models, ens_costs = [], []

        print(f'\n{"="*60}')
        print(f'ENSEMBLE (k={gk[0]},m={gk[1]}): {ENSEMBLE_SIZE} models')
        print(f'{"="*60}')

        for eidx, (h_dim, n_blk, drop, lr, noise_std, seed_off) in enumerate(ENSEMBLE_CONFIGS):
            print(f'\n  Ens {eidx+1}/{ENSEMBLE_SIZE} (h={h_dim}, b={n_blk}, d={drop})')
            seed = seed_off + gk[0] * 100 + gk[1] * 10
            model = ExpertDNN(info['feature_dim'], h_dim, n_blk, drop).to(device)
            val_loss, state = train_single_run(
                model, info['train_loader'], info['val_loader'],
                info['train_size'], info['val_size'], device,
                lr=lr, weight_decay=5e-4,
                max_epochs=ENSEMBLE_EPOCHS,
                patience=80, noise_std=noise_std,
                use_combined_loss=True, seed=seed,
            )
            model.load_state_dict(state); model.to(device); model.eval()
            ens_models.append(model)
            ens_costs.append(val_loss)
            print(f'    ✓ Best: {val_loss:.6f}')

        weak_ensembles[gk] = ens_models
        ensemble_val_costs[gk] = ens_costs
else:
    print('No weak groups — skipping ensemble.')

In [ ]:
# ============================================================
# Section 12: Evaluate All Strategies → Pick Best per Group
# ============================================================

improved_costs = {}
ensemble_weights = {}
best_strategy = {}
use_ensemble_for = []

if weak_groups:
    for gk in weak_groups:
        k_val, m_val = gk

        # Raw val features from ORIGINAL split
        orig_indices = [i for i, s in enumerate(X_combined) if s[1] == k_val and s[2] == m_val]
        X_grp = X_array[orig_indices]
        y_grp = y_log2[orig_indices]
        np.random.seed(42)
        perm = np.random.permutation(len(X_grp))
        split = int(0.9 * len(perm))
        val_idx = perm[split:]
        X_val_raw = X_grp[val_idx]
        y_val = y_grp[val_idx].flatten()

        # A) Original expert
        orig_scaler = group_data[gk]['scaler']
        orig_model = experts[gk]; orig_model.eval()
        t_orig = torch.tensor(orig_scaler.transform(X_val_raw), dtype=torch.float32).to(device)
        orig_preds = batch_predict(orig_model, t_orig)
        orig_cost = ((y_val - orig_preds) ** 2).mean()

        # B) Retrained expert
        ret_preds = None; ret_cost = float('inf')
        if gk in retrained_experts:
            ret_scaler = retrained_scalers[gk]; ret_model = retrained_experts[gk]; ret_model.eval()
            t_ret = torch.tensor(ret_scaler.transform(X_val_raw), dtype=torch.float32).to(device)
            ret_preds = batch_predict(ret_model, t_ret)
            ret_cost = ((y_val - ret_preds) ** 2).mean()

        # C) Ensemble
        ens_pred_arrays = []
        if gk in weak_ensembles:
            ens_scaler = weak_scalers[gk]
            t_ens = torch.tensor(ens_scaler.transform(X_val_raw), dtype=torch.float32).to(device)
            for em in weak_ensembles[gk]:
                em.eval()
                ens_pred_arrays.append(batch_predict(em, t_ens))

        pool_no_orig = ([ret_preds] if ret_preds is not None else []) + ens_pred_arrays
        pool_with_orig = [orig_preds] + pool_no_orig

        results = {'original': orig_cost}
        if ret_preds is not None:
            results['retrained'] = ret_cost

        w = None; si = None  # for storage later
        if pool_no_orig:
            arr_full = np.array(pool_with_orig)
            arr_no = np.array(pool_no_orig)
            results['simple'] = ((y_val - np.mean(arr_no, axis=0)) ** 2).mean()
            results['full'] = ((y_val - np.mean(arr_full, axis=0)) ** 2).mean()
            results['median'] = ((y_val - np.median(arr_full, axis=0)) ** 2).mean()

            pool_costs = [orig_cost]
            if ret_preds is not None: pool_costs.append(ret_cost)
            pool_costs.extend(ensemble_val_costs.get(gk, [1.0]*len(ens_pred_arrays)))
            inv = [1.0/(c+1e-8) for c in pool_costs]
            w = np.array([x/sum(inv) for x in inv])
            results['weighted'] = ((y_val - np.average(arr_full, axis=0, weights=w)) ** 2).mean()

            K = min(5, len(pool_costs))
            si = np.argsort(pool_costs)[:K]
            tw = w[si]; tw = tw/tw.sum()
            results['topk'] = ((y_val - np.average(arr_full[si], axis=0, weights=tw)) ** 2).mean()

            for trim_n in [2, 3, 4]:
                if len(arr_full) > trim_n + 3:
                    per_m = [((y_val - arr_full[i]) ** 2).mean() for i in range(len(arr_full))]
                    keep = np.argsort(per_m)[:len(arr_full)-trim_n]
                    results[f'trimmed_{trim_n}'] = ((y_val - np.mean(arr_full[keep], axis=0)) ** 2).mean()

        best_name = min(results, key=results.get)
        best_val = results[best_name]

        print(f'\n  (k={k_val},m={m_val}): orig={orig_cost:.4f} → best={best_name}({best_val:.4f})')
        for name, val in sorted(results.items(), key=lambda x: x[1]):
            print(f'    {name:>15}: {val:.4f}')

        # Store keep indices for trimmed strategies
        trimmed_keep = None
        if best_name.startswith('trimmed_'):
            trim_n = int(best_name.split('_')[1])
            per_model_final = [((y_val - np.array(pool_with_orig)[i]) ** 2).mean()
                               for i in range(len(pool_with_orig))]
            trimmed_keep = np.argsort(per_model_final)[:len(pool_with_orig)-trim_n].tolist()
        elif best_name == 'full':
            trimmed_keep = list(range(len(pool_with_orig)))

        improved_costs[gk] = {
            'old_cost': float(orig_cost), 'new_cost': float(best_val),
            'strategy': best_name,
            'weights': w.tolist() if w is not None else None,
            'topk_indices': si.tolist() if si is not None else None,
            'topk_weights': tw.tolist() if si is not None else None,
            'trimmed_keep': trimmed_keep,
        }
        best_strategy[gk] = best_name
        ensemble_weights[gk] = w.tolist() if w is not None else []

    for gk in weak_groups:
        if improved_costs[gk]['new_cost'] < improved_costs[gk]['old_cost']:
            use_ensemble_for.append(gk)

    # --- Overall ---
    print('\n' + '-'*60)
    print('OVERALL EVALUATION:')
    total_new, total_cnt = 0.0, 0
    for gk in group_keys:
        if gk in improved_costs and improved_costs[gk]['new_cost'] < improved_costs[gk]['old_cost']:
            cost = improved_costs[gk]['new_cost']
            k_v, m_v = gk
            oi = [i for i, s in enumerate(X_combined) if s[1] == k_v and s[2] == m_v]
            ns = len(oi) - int(0.9 * len(oi))
            total_new += cost * ns; total_cnt += ns
            src = improved_costs[gk]['strategy']
        else:
            m_ = experts[gk]; m_.eval()
            pl, tl = [], []
            with torch.no_grad():
                for bx, by in group_data[gk]['val_loader']:
                    bx = bx.to(device); pl.append(m_(bx).cpu().numpy()); tl.append(by.numpy())
            pa, ta = np.concatenate(pl).flatten(), np.concatenate(tl).flatten()
            cost = ((ta-pa)**2).mean()
            total_new += ((ta-pa)**2).sum(); total_cnt += len(ta)
            src = 'original'; ns = len(ta)
        print(f'  (k={gk[0]},m={gk[1]}): cost={cost:.6f} [{src}] ({ns} samples)')
    new_overall = total_new / total_cnt
    print(f'\n  Base overall:  {overall_base:.6f}')
    print(f'  New overall:   {new_overall:.6f}')
    print(f'  Improvement:   {overall_base - new_overall:+.6f}')
else:
    print('No weak groups — base model is final.')

In [ ]:
# ============================================================
# Section 13: Save Model
# ============================================================

save_dict = {
    'feature_dim': FEATURE_DIM,
    'group_keys': group_keys,
    'ensemble_groups': use_ensemble_for,
    'ensemble_size': ENSEMBLE_SIZE if weak_groups else 0,
}

for gk in group_keys:
    ks = f'k{gk[0]}_m{gk[1]}'
    save_dict[f'expert_{ks}_state'] = experts[gk].state_dict()
    save_dict[f'scaler_{ks}'] = group_data[gk]['scaler']
    h, b, d = expert_hparams[gk]
    save_dict[f'expert_{ks}_hidden_dim'] = h
    save_dict[f'expert_{ks}_num_blocks'] = b
    save_dict[f'expert_{ks}_dropout'] = d

if weak_groups:
    for gk in use_ensemble_for:
        ks = f'k{gk[0]}_m{gk[1]}'
        strategy = best_strategy.get(gk, 'simple')
        save_dict[f'ensemble_{ks}_strategy'] = strategy

        if gk in retrained_experts:
            save_dict[f'retrained_{ks}_state'] = retrained_experts[gk].state_dict()
            save_dict[f'retrained_scaler_{ks}'] = retrained_scalers[gk]
            cfg = RETRAIN_CONFIGS[gk]
            save_dict[f'retrained_{ks}_hidden_dim'] = cfg[0]
            save_dict[f'retrained_{ks}_num_blocks'] = cfg[1]
            save_dict[f'retrained_{ks}_dropout'] = cfg[2]

        if gk in weak_ensembles:
            save_dict[f'ensemble_scaler_{ks}'] = weak_scalers[gk]
            for eidx, em in enumerate(weak_ensembles[gk]):
                save_dict[f'ensemble_{ks}_model{eidx}_state'] = em.state_dict()
                ea = ENSEMBLE_CONFIGS[eidx]
                save_dict[f'ensemble_{ks}_model{eidx}_hidden_dim'] = ea[0]
                save_dict[f'ensemble_{ks}_model{eidx}_num_blocks'] = ea[1]
                save_dict[f'ensemble_{ks}_model{eidx}_dropout'] = ea[2]

        if gk in ensemble_weights:
            save_dict[f'ensemble_{ks}_weights'] = ensemble_weights[gk]
        if gk in improved_costs:
            ic = improved_costs[gk]
            if ic.get('topk_indices'):
                save_dict[f'ensemble_{ks}_topk_indices'] = ic['topk_indices']
                save_dict[f'ensemble_{ks}_topk_weights'] = ic['topk_weights']
            if ic.get('trimmed_keep'):
                save_dict[f'ensemble_{ks}_trimmed_keep'] = ic['trimmed_keep']

model_path = os.path.join(OUTPUT_DIR, 'moe_model.pth')
torch.save(save_dict, model_path)
file_size = os.path.getsize(model_path) / (1024 * 1024)
print(f'Model saved to {model_path} ({file_size:.1f} MB)')
print(f'  {len(group_keys)} base experts')
if use_ensemble_for:
    print(f'  {len(use_ensemble_for)} ensemble groups × {ENSEMBLE_SIZE} models')
    for gk in use_ensemble_for:
        print(f'    (k={gk[0]},m={gk[1]}): {best_strategy.get(gk, "simple")}')

In [ ]:
# ============================================================
# Section 14: TEST — Load Model & Predict (self-contained)
# ============================================================
# This cell is fully self-contained: it reloads the checkpoint
# from disk so the same logic works on a fresh kernel.
# ============================================================

import pickle, numpy as np, torch, torch.nn as nn, os
from collections import defaultdict

test_file_path  = os.path.join(DATA_DIR, 'CSCE-636-Project-1-Train-n_k_m_P')
model_file_path = os.path.join(OUTPUT_DIR, 'moe_model.pth')
output_file_path = os.path.join(OUTPUT_DIR, 'predicted_mHeights')

# --- Architecture (must match training) ---
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()
    def forward(self, x): return self.relu(self.block(x) + x)

class ExpertDNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=3, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
        )
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)]
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1),
        )
    def forward(self, x):
        x = self.input_proj(x); x = self.res_blocks(x); return self.output_head(x)

def extract_features(sample):
    n, k, m, P = sample
    max_k, max_nk, max_sv = 6, 5, 5
    P_padded = np.zeros((max_k, max_nk)); P_padded[:k, :n-k] = P; P_flat = P_padded.flatten()
    row_norms = np.zeros(max_k)
    for i in range(k): row_norms[i] = np.linalg.norm(P[i, :])
    col_norms = np.zeros(max_nk)
    for j in range(n-k): col_norms[j] = np.linalg.norm(P[:, j])
    sv = np.zeros(max_sv)
    try: svs = np.linalg.svd(P, compute_uv=False); sv[:len(svs)] = np.sort(svs)[::-1]
    except: pass
    frob_norm = np.linalg.norm(P, 'fro')
    max_abs = np.max(np.abs(P)); mean_abs = np.mean(np.abs(P))
    std_abs = np.std(np.abs(P)); min_abs = np.min(np.abs(P))
    try: cond = np.log1p(np.linalg.cond(P))
    except: cond = 0.0
    if not np.isfinite(cond): cond = 50.0
    mean_row = np.mean(row_norms[:k]) if k > 0 else 1.0
    mean_col = np.mean(col_norms[:n-k]) if (n-k) > 0 else 1.0
    ratio = mean_row / (mean_col + 1e-8)
    G = np.hstack([np.eye(k), P]); g_col_norms = np.zeros(9)
    for j in range(n): g_col_norms[j] = np.linalg.norm(G[:, j])
    g_col_max = np.max(g_col_norms); g_col_min = np.min(g_col_norms[:n])
    g_col_mean = np.mean(g_col_norms[:n]); g_col_std = np.std(g_col_norms[:n])
    try: rank = np.linalg.matrix_rank(P)
    except: rank = min(k, n-k)
    effective_rank = np.sum(sv > 1e-6)
    sv_ratio = sv[0] / (sv[min(k,n-k)-1]+1e-10) if sv[min(k,n-k)-1] > 1e-10 else 100.0
    sv_sum = np.sum(sv); sv_energy_ratio = sv[0] / (sv_sum + 1e-10)
    try: PtP = P.T @ P; ptP_trace = np.trace(PtP); ptP_frob = np.linalg.norm(PtP, 'fro')
    except: ptP_trace = frob_norm**2; ptP_frob = frob_norm**2
    try: sq = min(k, n-k); det_val = np.log1p(abs(np.linalg.det(P[:sq, :sq])))
    except: det_val = 0.0
    if not np.isfinite(det_val): det_val = 0.0
    return np.concatenate([[n,k,m], P_flat, row_norms, col_norms, sv,
        [frob_norm, max_abs, mean_abs, std_abs, min_abs], [cond, ratio],
        g_col_norms, [g_col_max, g_col_min, g_col_mean, g_col_std],
        [rank, effective_rank], [sv_ratio, sv_sum, sv_energy_ratio],
        [ptP_trace, ptP_frob], [det_val]])

def batch_predict(model, tensor, batch_size=2048):
    preds = []; model.eval()
    with torch.no_grad():
        for bi in range(0, len(tensor), batch_size):
            preds.append(model(tensor[bi:bi+batch_size]).cpu().numpy().flatten())
    return np.concatenate(preds)

# --- Load ---
device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
checkpoint = torch.load(model_file_path, map_location=device, weights_only=False)
feature_dim = checkpoint['feature_dim']
group_keys_ck = checkpoint['group_keys']
ensemble_groups = checkpoint.get('ensemble_groups', [])
ensemble_size = checkpoint.get('ensemble_size', 0)

experts_ck, scalers_ck = {}, {}
for gk in group_keys_ck:
    ks = f'k{gk[0]}_m{gk[1]}'
    h = checkpoint.get(f'expert_{ks}_hidden_dim', 256)
    b = checkpoint.get(f'expert_{ks}_num_blocks', 3)
    d = checkpoint.get(f'expert_{ks}_dropout', 0.15)
    m_ = ExpertDNN(feature_dim, h, b, d).to(device)
    m_.load_state_dict(checkpoint[f'expert_{ks}_state']); m_.eval()
    experts_ck[gk] = m_; scalers_ck[gk] = checkpoint[f'scaler_{ks}']

retrained_ck, retrained_sc_ck = {}, {}
ens_experts_ck, ens_scalers_ck = {}, {}
ens_strategies, ens_weights_ck, ens_topk_ck, ens_trimmed_ck = {}, {}, {}, {}

for gk in ensemble_groups:
    ks = f'k{gk[0]}_m{gk[1]}'
    ens_strategies[gk] = checkpoint.get(f'ensemble_{ks}_strategy', 'simple')
    if f'retrained_{ks}_state' in checkpoint:
        h = checkpoint.get(f'retrained_{ks}_hidden_dim', 512)
        b = checkpoint.get(f'retrained_{ks}_num_blocks', 5)
        d = checkpoint.get(f'retrained_{ks}_dropout', 0.20)
        rm = ExpertDNN(feature_dim, h, b, d).to(device)
        rm.load_state_dict(checkpoint[f'retrained_{ks}_state']); rm.eval()
        retrained_ck[gk] = rm
        retrained_sc_ck[gk] = checkpoint.get(f'retrained_scaler_{ks}', scalers_ck[gk])
    models_ck = []
    for ei in range(ensemble_size):
        sk = f'ensemble_{ks}_model{ei}_state'
        if sk in checkpoint:
            h = checkpoint.get(f'ensemble_{ks}_model{ei}_hidden_dim', 512)
            b = checkpoint.get(f'ensemble_{ks}_model{ei}_num_blocks', 5)
            d = checkpoint.get(f'ensemble_{ks}_model{ei}_dropout', 0.18)
            em = ExpertDNN(feature_dim, h, b, d).to(device)
            em.load_state_dict(checkpoint[sk]); em.eval()
            models_ck.append(em)
    ens_experts_ck[gk] = models_ck
    ens_scalers_ck[gk] = checkpoint.get(f'ensemble_scaler_{ks}', scalers_ck[gk])
    ens_weights_ck[gk] = checkpoint.get(f'ensemble_{ks}_weights', None)
    ens_topk_ck[gk] = {
        'indices': checkpoint.get(f'ensemble_{ks}_topk_indices', None),
        'weights': checkpoint.get(f'ensemble_{ks}_topk_weights', None),
    }
    ens_trimmed_ck[gk] = checkpoint.get(f'ensemble_{ks}_trimmed_keep', None)

print(f'Loaded {len(experts_ck)} experts on {device}')
print(f'Retrained: {list(retrained_ck.keys())}')
print(f'Ensemble: {list(ens_experts_ck.keys())} ({ensemble_size} each)')
for gk in ensemble_groups:
    print(f'  (k={gk[0]},m={gk[1]}): {ens_strategies.get(gk,"simple")}')

# --- Load test data ---
with open(test_file_path, 'rb') as f: test_data = pickle.load(f)
print(f'Loaded {len(test_data)} test samples.')

test_groups = defaultdict(list)
for i, sample in enumerate(test_data):
    n, k, m, P = sample
    test_groups[(k, m)].append((i, extract_features(sample)))

# --- Predict ---
predicted_mheights = [0.0] * len(test_data)
for gk, items in test_groups.items():
    indices = [it[0] for it in items]
    features = np.array([it[1] for it in items], dtype=np.float32)
    if gk not in experts_ck:
        for idx in indices: predicted_mheights[idx] = 1.0
        continue

    orig_sc = scalers_ck[gk].transform(features)
    orig_t = torch.tensor(orig_sc, dtype=torch.float32).to(device)
    orig_preds = batch_predict(experts_ck[gk], orig_t)

    if gk in ens_experts_ck and ens_experts_ck[gk]:
        strategy = ens_strategies.get(gk, 'simple')
        all_pred_arrays = []
        if gk in retrained_ck:
            ret_sc = retrained_sc_ck[gk].transform(features)
            ret_t = torch.tensor(ret_sc, dtype=torch.float32).to(device)
            all_pred_arrays.append(batch_predict(retrained_ck[gk], ret_t))
        ens_sc = ens_scalers_ck[gk].transform(features)
        ens_t = torch.tensor(ens_sc, dtype=torch.float32).to(device)
        for em in ens_experts_ck[gk]:
            all_pred_arrays.append(batch_predict(em, ens_t))
        arr = np.array(all_pred_arrays)

        if strategy == 'weighted' and ens_weights_ck.get(gk) is not None:
            w_ck = np.array(ens_weights_ck[gk])
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.average(full_arr, axis=0, weights=w_ck[:len(full_arr)])
        elif strategy == 'topk' and ens_topk_ck.get(gk, {}).get('indices') is not None:
            tk = ens_topk_ck[gk]
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            ti = [i for i in tk['indices'] if i < len(full_arr)]
            tw_ck = np.array(tk['weights'][:len(ti)]); tw_ck = tw_ck/tw_ck.sum()
            preds_log2 = np.average(full_arr[ti], axis=0, weights=tw_ck)
        elif strategy.startswith('trimmed') and ens_trimmed_ck.get(gk) is not None:
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            keep_idx = [i for i in ens_trimmed_ck[gk] if i < len(full_arr)]
            preds_log2 = np.mean(full_arr[keep_idx], axis=0)
        elif strategy == 'median':
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.median(full_arr, axis=0)
        elif strategy == 'full':
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.mean(full_arr, axis=0)
        else:
            preds_log2 = np.mean(arr, axis=0)
        source = f'ens-{strategy}({len(all_pred_arrays)})'
    else:
        preds_log2 = orig_preds; source = 'single'

    preds = np.maximum(2.0 ** preds_log2, 1.0)
    for idx, val in zip(indices, preds): predicted_mheights[idx] = float(val)
    print(f'  (k={gk[0]},m={gk[1]}) [{source:>25}]: {len(indices)} samples, '
          f'range=[{preds.min():.2f}, {preds.max():.2f}]')

with open(output_file_path, 'wb') as f:
    pickle.dump(predicted_mheights, f)

print(f'\nPredictions saved to {output_file_path}')
print(f'Total: {len(predicted_mheights)}')
print(f'Range: [{min(predicted_mheights):.4f}, {max(predicted_mheights):.4f}]')
print(f'All >= 1: {all(h >= 1.0 for h in predicted_mheights)}')
print('\n🎉 DONE!')